# Credit-risk early warning: why a "great" feature can be a leak
**Project 2 - credit-risk-pd** | Feature Engineering & MLOps, Semester VII

Run the cells top to bottom (about 1 minute; the first run generates the data). Every cell calls code that lives in `src/`, the same code the pipeline, the tests and the API use.

## 1. Learning objectives
After this notebook you can
1. explain the *prediction timestamp* (`as_of_date`) and the *availability rule* `available_at <= as_of_date`;
2. compute the core behavioural features from transactions and check them by hand;
3. show how one leaky feature inflates AUC and then collapses in production, and prove a pipeline is safe with the **future-perturbation test**;
4. build a time-based split, a logistic baseline and a gradient-boosting candidate, and read the metrics as *business* numbers;
5. run a drift check (KS test) and explain why drift alone does not mean "retrain".

## 2. Business problem
A bank wants a **monthly early-warning list**. On every month-end (`as_of_date`) it estimates, for each *existing* customer, the probability of **default (90+ days past due) within the next 6 months**. Collections can call only the riskiest **10 %** of customers, so what matters is how many real defaulters are inside that top 10 %.

## 3. Dataset (synthetic, seeded)
| table | rows describe | key columns |
|---|---|---|
| `customers` | one row per customer | signup_date, segment, loan_amount, credit_limit |
| `transactions` | every account movement | txn_date, amount, txn_type, balance_after |
| `loan_status` | one row per customer-month | missed_payment, days_past_due, **available_at** (published on day 5 of the next month), collections_status, recovery_amount |
| `defaults` | the outcome | default_date (empty = never) |

The **snapshot table** has one row per (customer, month-end) with 9 behavioural features, 3 derived features and the target `default_next_6m`.

## 4. Why this concept is needed: the prediction-time question
> *"Would this number really exist on the day we score the customer?"*

After a customer defaults the bank sweeps the account, so the **balance collapses** and recoveries appear. A feature that accidentally looks at that period looks magical in the lab and useless in production.

## 5. Imports and setup

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)                      # all src modules use paths relative to the project root
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

from src.common.paths import DATA_RAW, SNAPSHOTS_PATH
print("project root:", ROOT)

## 6. Load data (generated on the first run)

In [ ]:
if not (DATA_RAW / "transactions.csv").exists() or not SNAPSHOTS_PATH.exists():
    from src.data import make_dataset
    from src.features import build_features
    make_dataset.main()                 # raw tables (about 5 seconds)
    build_features.main()               # leakage-safe snapshots + production batches (about 10 seconds)

from src.features.build_features import load_raw, add_derived_features
from src.features.definitions import BASE_FEATURES, MODEL_FEATURES, TARGET
raw = load_raw()
snap = add_derived_features(pd.read_csv(SNAPSHOTS_PATH, parse_dates=["as_of_date"]))
for name, df in raw.items():
    print(f"{name:<13}{len(df):>9,} rows   columns: {list(df.columns)}")

## 7. Explore the data

In [ ]:
raw["transactions"].head(8)

In [ ]:
split = snap.groupby("split").agg(rows=(TARGET, "size"), defaulters=(TARGET, "sum"), default_rate=(TARGET, "mean"),
                                  first=("as_of_date", "min"), last=("as_of_date", "max")).loc[["train", "purge", "val", "test", "production"]]
split["default_rate"] = split["default_rate"].round(4)
split

Things to notice:
* **Imbalance**: about 4 % defaulters. "Nobody defaults" is 96 % accurate and useless, so we will not use accuracy.
* **Time split, not random**: train < purge < val < test < production. The *purge* months exist because the label of a train row is known only 6 months later.
* Production default rate is higher: a simulated macro stress started in 2023-09.

## 8. Baseline: logistic regression on the approved features

In [ ]:
import yaml
from src.models.pipelines import make_logistic, make_hgb
from src.common.config import load_params
params = load_params()
features = yaml.safe_load(open("configs/selected_features.yaml"))["selected"]
tr, va, te = (snap[snap["split"] == s] for s in ("train", "val", "test"))
print(len(features), "features:", features)
baseline = make_logistic(features, params["model"]["logistic"]).fit(tr[features], tr[TARGET])
print("baseline  validation ROC-AUC:", round(roc_auc_score(va[TARGET], baseline.predict_proba(va[features])[:, 1]), 4))

## 9. Concept demonstration: the availability rule
`loan_status` for month M is published on day 5 of month M+1 (`available_at`). The helper below is the whole rule.

In [ ]:
from src.features.build_features import filter_available
ls = raw["loan_status"]
feb = ls[ls["month"] == "2023-02-28"]
print("February rows available_at:", feb["available_at"].iloc[0].date())
print("usable on 2023-02-28 :", len(filter_available(feb, "2023-02-28", "available_at")), "rows")
print("usable on 2023-03-31 :", len(filter_available(feb, "2023-03-31", "available_at")), "rows")

## 10. Manual calculation: one tiny customer, features by hand vs code
`src/features/manual_examples.py` builds a toy customer with round numbers (salary in 5 of 6 months, one missed instalment, balance falling 5,000 per month). We compute the features with the **real** function and compare with the arithmetic done by hand.

In [ ]:
from src.features.manual_examples import tiny_customer, AS_OF
from src.features.build_features import build_snapshot_features
t = tiny_customer()
f = build_snapshot_features(t["customers"], t["transactions"], t["loan_status"], AS_OF).iloc[0]
by_hand = {"salary_credit_frequency": 5/6, "loan_payment_ratio": 5/6, "missed_payment_count": 1, "credit_utilization": 0.2,
           "income_stability": np.sqrt(50)/100, "recent_transaction_count": 4, "average_balance": 112000.0, "balance_trend": -5000.0, "account_age": 36}
cmp = pd.DataFrame({"by_hand": by_hand, "code": {k: float(f[k]) for k in by_hand}})
cmp["match"] = np.isclose(cmp["by_hand"], cmp["code"], atol=1e-4)
cmp.round(4)

Balance trend on three points, the textbook way: balances `[100, 80, 60]` at x = 0, 1, 2 give x_mean = 1, y_mean = 80, slope = ((-1)(20) + 0 + (1)(-20)) / (1 + 0 + 1) = **-20**.
Income stability = std / mean of salary amounts: salaries 100, 110, 90, 100, 100 give std 7.07 and mean 100, so **0.0707** (a regular salary has a value near 0).

In [ ]:
from src.features.build_features import _slope_6
print("slope of [100, 80, 60] =", _slope_6(np.array([[100.0, 80.0, 60.0]]))[0])

## 11. Python implementation: prove the pipeline is safe (future-perturbation test)

In [ ]:
from src.features.leakage_checks import future_perturbation_test
from src.features.leakage_demo import LEAKY_BUILDERS

def safe_builder(c, t, l, a):
    return build_snapshot_features(c, t, l, a)

res = {"build_snapshot_features (safe)": future_perturbation_test(safe_builder, raw, "2022-06-30")["passed"]}
for name, fn in LEAKY_BUILDERS.items():
    res[name] = future_perturbation_test(fn, raw, "2022-06-30")["passed"]
pd.Series(res, name="features unchanged when the FUTURE is corrupted?")

The test overwrites every transaction and loan_status row that is **not yet available** on 2022-06-30 with garbage and rebuilds the features.
A leakage-safe pipeline returns identical numbers (`True`). The four leaky features change (`False`): they read the future.

## 12. Visualization: what the leaky feature sees

In [ ]:
tx = raw["transactions"]; de = raw["defaults"].dropna()
cid = int(de[de["default_date"].between("2022-09-01", "2022-12-31")].iloc[0]["customer_id"])
d_date = de.set_index("customer_id").loc[cid, "default_date"]
c = tx[(tx["customer_id"] == cid) & tx["txn_date"].between(d_date - pd.Timedelta(days=330), d_date + pd.Timedelta(days=180))]
as_of = d_date - pd.DateOffset(months=4) + pd.offsets.MonthEnd(0)
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(c["txn_date"], c["balance_after"], lw=1.2)
ax.axvline(as_of, color="green", ls="--", label=f"as_of_date {as_of.date()} (we score here)")
ax.axvline(d_date, color="red", ls="--", label=f"default {d_date.date()}")
ax.axvspan(as_of, as_of + pd.DateOffset(months=6), color="red", alpha=0.08, label="6-month target window")
ax.set_title(f"customer {cid}: balance collapses AFTER default"); ax.set_ylabel("balance_after"); ax.legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.show()

## 13. Model: gradient-boosting candidate and the leakage comparison

In [ ]:
candidate = make_hgb(features, params["model"]["hgb"]).fit(tr[features], tr[TARGET])
auc = lambda m, d: round(roc_auc_score(d[TARGET], m.predict_proba(d[features])[:, 1]), 4)
print("validation ROC-AUC   baseline:", auc(baseline, va), "  candidate:", auc(candidate, va))

In [ ]:
# the leaky feature, computed for ONE test month (as_of 2022-12-31), compared with the honest feature on the same customers
from src.features.leakage_demo import average_balance_leaky
as_of = pd.Timestamp("2022-12-31")
one = snap[snap["as_of_date"] == as_of].merge(average_balance_leaky(raw["customers"], raw["transactions"], raw["loan_status"], as_of), on="customer_id", how="left")
one["average_balance_leaky"] = one["average_balance_leaky"].fillna(0)
print("single-feature AUC  honest average_balance :", round(roc_auc_score(one[TARGET], -one["average_balance"]), 3))
print("single-feature AUC  LEAKY  average_balance :", round(roc_auc_score(one[TARGET], -one["average_balance_leaky"]), 3), " <- too good to be true")

Full numbers (model trained with the leaky column: test AUC about 0.96, live AUC about 0.78, below the honest model) are in `python -m src.features.leakage_demo` and `docs/sample_outputs/04_leakage_demo.txt`.
**A jump of 0.07 AUC from one column should make you suspicious, not happy.**

## 14. Evaluation: metrics as business numbers (test period, never used for decisions)

In [ ]:
from src.models.evaluate import compute_metrics, business_summary, print_business_summary
p = candidate.predict_proba(te[features])[:, 1]
print(pd.Series(compute_metrics(te[TARGET], p)))
print()
print_business_summary(business_summary(te[TARGET], p, 0.10))

## 15. Interpretation: permutation importance (association, not causation)

In [ ]:
r = permutation_importance(candidate, te[features], te[TARGET], scoring="roc_auc", n_repeats=5, random_state=0)
pd.Series(r.importances_mean, index=features).sort_values(ascending=False).round(4)

Shuffling a column and measuring the AUC drop shows what the model *relies on*. Correlated features (`missed_payment_count` and `loan_payment_ratio`) share importance. Importance is not causality: lowering someone's utilisation does not by itself lower their default risk.

## 16. Production: drift check with the KS test, then the decision ladder

In [ ]:
from src.monitoring.run_drift_check import run
ref = "data/processed/reference.csv"
res = run("data/production/batch_drifted.csv", ref, today="2024-02-29", scheduled=False, alpha=0.05, min_effect=0.10, quiet=True)
t = pd.DataFrame(res["drift"])[["feature", "ks_statistic", "p_value", "status"]]
print("H0: reference and production values come from the same distribution (alpha = 0.05; 'large' means D >= 0.10)")
t.round(4)

In [ ]:
print("data quality ok:", res["data_quality"]["ok"])
print("DECISION on 2024-02-29:", res["decision"]["decision"])
for r in res["decision"]["reasons"]:
    print("  *", r)

Four key features drifted, but on 2024-02-29 the outcome of the January customers is **unknown for 6 more months** (label lag). The ladder says **INVESTIGATE**, not retrain. Later, with early-delinquency evidence or real labels, the same batch becomes a `RETRAIN_CANDIDATE`:

In [ ]:
res2 = run("data/production/batch_drifted.csv", ref, today="2025-06-30", scheduled=False, alpha=0.05, min_effect=0.10, quiet=True)
print("DECISION on 2025-06-30:", res2["decision"]["decision"])
print("labelled AUC drop:", res2["labelled"]["auc_drop"], "| early-delinquency ratio:", res2["pseudo_label"]["rate_ratio"])

## 17. Common mistakes
* Building the training table at the **extract date** instead of at `as_of_date` (the leak you just saw).
* Using columns that exist *because* the customer defaulted (`recoveries`, `collections_status`, `days_past_due` of the target period).
* A **random** train/test split of monthly snapshots: the same customer is on both sides and the future helps the past.
* Fitting imputers/scalers on all data. Here they live inside the sklearn Pipeline and learn from the training rows only.
* Reading "drift detected" as "retrain now". Drift is a signal; data quality, outcome evidence and a human approval come next.

## 18. Industry connection
| role | question in this project |
|---|---|
| Data engineer | Is `loan_status` really published on day 5? What happens when the source system changes? |
| Data scientist | Which features survive the leakage test and the out-of-time check? |
| ML engineer | Does the monthly batch job reproduce the training features exactly (no skew)? |
| Model risk / governance | Is every feature owned, approved, versioned? Who signed off the new champion? |

## 19. Exercises
1. In `tiny_customer()` change January so that the salary *is* paid. Predict, then check, the new `salary_credit_frequency`, `income_stability` and `missed_payment_count`.
2. Move the leaky window to `as_of + 3 months`. Does the single-feature AUC rise or fall? Why?
3. Re-run the baseline with `account_age` added. Why did the selection step veto it?
4. Change `C` of the logistic regression. Which ranking metric moves most?
5. Which two features would you put on a "watch list" if `income_stability` had a 30 % null rate next month?

## 20. Key takeaways
* A feature belongs to a **timestamp**: compute it only from rows with `available_at <= as_of_date`.
* **Prove** safety with the future-perturbation test; **block** unapproved features in training with the registry.
* Judge a credit model by **recall in the top 10 %**, lift and PR-AUC, on a **later** period.
* Drift is a signal. Decide with data quality + drift + outcome evidence, then compare champion and challenger on the **same** newer hold-out, then ask a human.